# Diff attribution graphs before and after RL

48 graphs of Qwen3-1.7B from the RL-hack-circuit study: 8 Pigment Gym prompts, before and after RL, for the run that learned to stuff comments with reward keywords (`installed`), another seed that also hacked (`replication`) and a run that did not (`control`).

In [1]:
from circuitkit.diff import diff_graphs, load_summaries

g = load_summaries("data/rl_hack_graphs.json.gz")
{k: len(v) for k, v in g.items()}

{'installed/before': 8,
 'installed/after': 8,
 'replication/before': 8,
 'replication/after': 8,
 'control/before': 8,
 'control/after': 8}

In [2]:
rep = diff_graphs(
    g["installed/before"],
    g["installed/after"],
    control=(g["control/before"], g["control/after"]),
    replication=(g["replication/before"], g["replication/after"]),
    fvu_before=g["installed/before"][0].meta["fvu_by_layer"],
    fvu_after=g["installed/after"][0].meta["fvu_by_layer"],
)
for c in rep.top:
    print(f"{c.unit:>10} {c.status:7} delta={c.delta:+.4f} specific={c.specific} replicated={c.replicated}")

 21:154092 up      delta=+0.0035 specific=True replicated=True
 16:146794 up      delta=+0.0026 specific=True replicated=True
  16:58137 up      delta=+0.0021 specific=True replicated=True
  26:88470 up      delta=+0.0018 specific=True replicated=False
 26:162002 up      delta=+0.0017 specific=True replicated=False
 19:143996 up      delta=+0.0015 specific=False replicated=False
 19:141589 up      delta=+0.0015 specific=True replicated=True
 25:137808 up      delta=+0.0014 specific=True replicated=False
  19:78662 up      delta=+0.0013 specific=True replicated=True
 22:160272 gained  delta=+0.0013 specific=True replicated=True


Is the change real? The swap null, the split-half check and the validity check:

In [3]:
s = rep.summary
print(f"total change {s.change_score:.3f} vs {s.change_null_mean:.3f} under the null, p = {s.change_p:.3f}")
print(f"split-half top-10 overlap {s.split_half_overlap:.0%} (chance {s.split_half_chance:.1%})")
print("validity ok:", rep.validity.ok, rep.validity.warnings)

total change 0.333 vs 0.111 under the null, p = 0.012
split-half top-10 overlap 70% (chance 0.2%)
validity ok: True []


In [4]:
print(rep.caveats[0])

The transcoders were trained on the base model. On a fine-tuned model they may reconstruct the MLPs less well, and a feature that 'appears' may be the old dictionary describing new computation. Read the validity section: if reconstruction error or error-node influence rises after training, treat gained and lost features as candidates to test (for example with circuitkit.validate), not as findings.


In [5]:
from circuitkit.report import render_diff, save_html

save_html(render_diff(rep), "reports/diff_rl_hack.html")

PosixPath('reports/diff_rl_hack.html')